<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 2 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">观察存储和写入批次</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">使用统一订单样本，观察 SQL、结果与验收证据。请按顺序运行单元。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Doris 4.1.3 target · Order data · Isolated course database</span>
</div>

By the end of this lab, you will have a running Doris environment, an `events` table containing more than 10 million rows, and analytical results produced from that table. Run the cells in order.

[讲义](course2_doris_architecture.md) · [课程入口](../README.md)


## 实验范围

仅重建 d02_batch 和 d02_small。小样本用于解释观测方法，不证明性能提升；后台 Compaction 可能很快消除版本差异。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = WarehouseLab()




## 1. 对齐变量

两张表使用完全相同的 Schema、数据和桶数。关闭会合并小事务的 Group Commit，再比较一次批量写入和逐行写入。


In [ ]:
lab.execute("DROP TABLE IF EXISTS d02_batch")
ddl = order_ddl("d02_batch")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS d02_small")
ddl = order_ddl("d02_small")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
rows = order_rows(fixture("orders.json"))
lab.execute("SET group_commit = 'off_mode'")
lab.insert("d02_batch", ORDER_COLUMNS, rows)
for row in rows:
    lab.insert("d02_small", ORDER_COLUMNS, [row])


## 2. 查询元数据

SHOW TABLETS 中的 VersionCount 是版本相关观测，不等于直接数 Rowset。记录采样时刻、事务次数和后台合并影响。Rowset/Segment 深入检查需要受控管理接口，首版不自动调用。


In [ ]:
lab.sql("SHOW CREATE TABLE d02_batch");
lab.sql("SHOW PARTITIONS FROM d02_batch");
lab.sql("SHOW TABLETS FROM d02_batch");
lab.sql("SHOW TABLETS FROM d02_small");


## 3. 先确认业务结果没有变化

两张表都应为 10 行、1400.00。不把 VersionCount 必然更高或运行更快写成断言。


In [ ]:
for table in ("d02_batch", "d02_small"):
    expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {table}"), [(10, "1400.00")])
lab.sql("EXPLAIN SELECT order_id, order_amount FROM d02_batch WHERE order_id = 1001");
lab.close()


## 待补的录制实验

放大数据的扫描量对照、Query Profile 展示、Rowset 管理接口和持续版本积压演示尚未实现；不能由这个小样本推导吞吐结论。
